## Model Selection

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import HistGradientBoostingRegressor

from features import build_input, TEST_DB, FULL_DB, FEATURES, FUNDAMENTAL_FEATURES
from course_data import date_folds, make_pipeline, prediction_metrics # lecture code

cache = Path("outputs/panel_full.parquet")
if cache.exists():
    panel = pd.read_parquet(cache)
else:
    cache.parent.mkdir(exist_ok=True)
    panel = build_input(FULL_DB, end="2026-09-30")            # ~5 min
    panel.to_parquet(cache)

assert set(FUNDAMENTAL_FEATURES) <= set(panel.columns), "outputs/panel_full.parquet"

panel.shape

(576243, 43)

In [2]:
DEV_START = "2010-01-01"
DEV_END = "2015-12-31"
LABEL_CUTOFF = "2016-03-31"
VALIDATION = (2012, 2013, 2014, 2015)

dev = panel.loc[panel.eligible
                & panel.date.between(DEV_START, DEV_END)
                & panel.target.notna()
                & panel.label_end.le(LABEL_CUTOFF)].reset_index(drop=True)
dev["target_rank"] = dev.groupby("date").target.rank(pct=True) - 0.5   # in the dev cell


print(len(dev))
display(dev.target.describe(percentiles=[.01, .5, .99]).round(3))
display(dev.nlargest(5, "target")[["date", "symbol", "target", "label_end"]])
display(dev[FEATURES].describe().T.round(3))


139554


count    139554.000
mean          1.209
std         320.986
min          -1.000
1%           -0.447
50%           0.029
99%           0.660
max      106847.564
Name: target, dtype: float64

,date,symbol,target,label_end
52214,2012-04-30,AAM,106847.563969,2012-07-30
28573,2011-04-29,AAM,54412.542926,2011-08-01
1748,2010-01-29,VBIV,816.214030,2010-05-03
5484,2010-03-31,VBIV,660.606916,2010-06-30
3611,2010-02-26,VBIV,656.212833,2010-06-01


,count,mean,std,min,25%,50%,75%,max
rsi_21,139403.0,52.020,13.939,0.0,42.439,52.040,61.697,100.000
dma_20,139554.0,0.004,0.068,-1.0,-0.023,0.005,0.031,9.567
dma_50,139554.0,0.012,0.121,-1.0,-0.033,0.013,0.056,24.658
dma_100,138685.0,0.024,0.150,-1.0,-0.040,0.025,0.088,19.338
momentum_252_21,135394.0,1.099,195.998,-1.0,-0.030,0.145,0.354,60513.372
volatility_63,139554.0,23.308,2183.949,0.0,0.219,0.302,0.438,271538.508


In [3]:
folds = date_folds(dev, validation_years=VALIDATION, train_start=DEV_START)
pd.DataFrame([{"year": yr, "train_rows": len(tr), "valid_rows": len(va),
               "last_train_label_end": dev.iloc[tr].label_end.max().date(),
               "first_valid_signal": dev.iloc[va].date.min().date()}
              for yr, (tr, va) in zip(VALIDATION, folds)])


,year,train_rows,valid_rows,last_train_label_end,first_valid_signal
0,2012,42341,23710,2012-01-30,2012-01-31
1,2013,66046,23804,2013-01-30,2013-01-31
2,2014,89841,23851,2014-01-30,2014-01-31
3,2015,111686,21902,2014-12-30,2015-01-30


In [4]:
def quintile_spread(frame, pred):
    """Mean (top 20% − bottom 20%) realized return, ranked within each date."""
    df = frame[["date", "target"]].assign(pred=np.asarray(pred))
    df["q"] = df.groupby("date").pred.rank(method="first", pct=True).mul(5).apply(np.ceil)
    by_date = df.groupby(["date", "q"]).target.mean().unstack()
    return (by_date[5] - by_date[1]).mean()


def evaluate(candidates, features, include_sector=False, rank=False, clip=(0.01, 0.99)):
    """Fit each model on each fold, y only between 1-99% quantile to filter out irregular returns"""
    X, y = dev[features], dev.target
    rows = []
    for name, model in candidates.items():
        pipe = make_pipeline(model, numeric_features=features, include_sector=include_sector)
        for yr, (tr, va) in zip(VALIDATION, folds):
            lo, hi = y.iloc[tr].quantile(list(clip))          # learned from train rows only
            y_fit = dev.target_rank.iloc[tr] if rank else y.iloc[tr].clip(lo, hi)
            fitted = clone(pipe).fit(X.iloc[tr], y_fit)
            pred = fitted.predict(X.iloc[va])
            valid = dev.iloc[va].assign(target=lambda d: d.target.clip(lo, hi))
            rows.append({"model": name, "year": yr, "sector": include_sector, "rank_target": rank,
                         **prediction_metrics(valid, pred).to_dict(),
                         "top_minus_bottom": quintile_spread(valid, pred)})
    return pd.DataFrame(rows)


In [5]:
candidates = {
    "training mean": DummyRegressor(),
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=100),
    "Lasso": Lasso(alpha=0.001, max_iter=10_000),
    "boosted trees": HistGradientBoostingRegressor(
        max_iter=100, max_leaf_nodes=15, min_samples_leaf=200,
        l2_regularization=10, early_stopping=False, random_state=42),
}
res_tech = evaluate(candidates, FEATURES).assign(features="technical", target="raw")
res_fund = evaluate(candidates, FEATURES + FUNDAMENTAL_FEATURES).assign(features="tech+fund", target="rank")
res_rank = evaluate({"Ridge": Ridge(alpha=100), "boosted trees": candidates["boosted trees"]},
                    FEATURES + FUNDAMENTAL_FEATURES, rank=True)
res_rank.pivot(index="model", columns="year", values="mean_daily_rank_IC").round(3)

for name, df in {"tech": res_tech, "fund": res_fund, "fund_rank": res_rank}.items():
    df.to_csv(f"outputs/results_{name}.csv", index=False)


In [6]:
allres = pd.concat([res_tech, res_fund, res_rank])
display(allres.pivot_table(index=["features", "target", "model"], columns="year",
                           values="mean_daily_rank_IC").round(3))
display(allres.groupby(["features", "target", "model"])[["RMSE", "mean_daily_rank_IC", "top_minus_bottom"]]
        .mean().round(4))


year                             2012   2013   2014   2015
features  target model                                    
tech+fund rank   Lasso         -0.011 -0.003 -0.008 -0.058
                 OLS           -0.009 -0.001 -0.002 -0.059
                 Ridge         -0.009 -0.001 -0.002 -0.059
                 boosted trees -0.018  0.015  0.032 -0.020
technical raw    Lasso         -0.008 -0.015 -0.030 -0.054
                 OLS           -0.007 -0.015 -0.026 -0.057
                 Ridge         -0.007 -0.015 -0.026 -0.057
                 boosted trees -0.002  0.003 -0.015 -0.026

RMSE  mean_daily_rank_IC  top_minus_bottom
features  target model                                                      
tech+fund rank   Lasso          0.1728             -0.0201           -0.0099
                 OLS            0.1736             -0.0178           -0.0093
                 Ridge          0.1736             -0.0179           -0.0090
                 boosted trees  0.1745              0.0022            0.0032
                 training mean  0.1708                 NaN           -0.0003
technical raw    Lasso          0.1725             -0.0267           -0.0120
                 OLS            0.1731             -0.0259           -0.0120
                 Ridge          0.1731             -0.0260           -0.0120
                 boosted trees  0.1741             -0.0101           -0.0010
                 training mean  0.1708                 NaN           -0.0003

In [7]:
alphas = evaluate({f"Ridge α={a:g}": Ridge(alpha=a) for a in [1, 1e3, 1e4, 1e5, 1e6]}, FEATURES)
alphas.groupby("model")[["RMSE", "mean_daily_rank_IC", "top_minus_bottom"]].mean() \
      .sort_values("mean_daily_rank_IC", ascending=False).round(4)


,RMSE,mean_daily_rank_IC,top_minus_bottom
model,,,
Ridge α=1e+06,0.1708,-0.0204,-0.0074
Ridge α=100000,0.1713,-0.0254,-0.0096
Ridge α=1,0.1731,-0.0259,-0.0120
Ridge α=1000,0.1729,-0.0270,-0.0120
Ridge α=10000,0.1722,-0.0299,-0.0125


In [8]:
pipe = make_pipeline(Ridge(alpha=100), numeric_features=FEATURES, include_sector=False)
tr, _ = folds[-1]
y_tr = dev.target.iloc[tr]
pipe.fit(dev[FEATURES].iloc[tr], y_tr.clip(*y_tr.quantile([.01, .99])))
pd.Series(pipe[-1].coef_[:len(FEATURES)], index=FEATURES).sort_values().round(4)


dma_100           -0.0185
dma_20            -0.0138
rsi_21            -0.0042
momentum_252_21   -0.0025
volatility_63      0.0026
dma_50             0.0218
dtype: float64

In [9]:
test = build_input(TEST_DB, end="2015-12-31")     # ~3 s
test[FUNDAMENTAL_FEATURES].describe().T

     30/30 symbols (3s)


,count,mean,std,min,25%,50%,75%,max
roe,2401.0,-7.075705,66.698756,-841.747284,0.091356,0.187069,0.287556,263.913324
debt_to_equity,2401.0,1.802667,8.999254,0.000000,0.250864,0.524891,1.325697,178.949420
book_to_px,2467.0,6.801562,122.157237,-0.000111,0.119321,0.253096,0.527141,3632.177819
earnings_yield,2467.0,117.851328,2509.098295,-1078.430661,0.017253,0.053821,0.075929,56627.274199
fcf_yield,2372.0,339.158030,7478.856064,-80.998976,0.020761,0.048901,0.073047,230770.945521
ebitda_ev,2446.0,-1.091753,38.689346,-460.977426,0.043657,0.089022,0.118328,882.402853
